To resolve dependency error install the correct version of langchain-anthropic to allign with langchain, langchain-core

In [0]:
# %pip index versions langchain-anthropic
%pip install langchain-anthropic==1.2.0

Always install the same version of langchain and langchain-core

In [0]:
# %pip index versions langchain
%pip install --upgrade langchain==1.2.10 langchain-core==1.2.10

In [0]:
import langchain.agents.middleware as mw
print("Available middleware components:")
print([x for x in dir(mw) if not x.startswith('_')])

## What is Middleware in LangChain Agents?

**Middleware** is a powerful pattern in `langchain.agents` that lets you **intercept, modify, and control** the behavior of agents at key points in their execution lifecycle — without changing the core agent logic.

Think of middleware as **pluggable layers** that wrap around your agent's model calls and tool calls:

```
User Request → [Middleware Layer 1] → [Middleware Layer 2] → ... → Agent Core → Response
```

### Middleware Hooks (Lifecycle Points)
| Hook | When it Fires | Use Case |
|------|--------------|----------|
| `before_model` | Before each LLM call | Modify prompts, add context, redact PII |
| `after_model` | After each LLM response | Log responses, validate output |
| `before_agent` | Before the agent loop starts | Initialize state, set limits |
| `after_agent` | After the agent loop ends | Cleanup, summarize results |
| `wrap_model_call` | Wraps the entire model call | Retry, fallback, rate limiting |
| `wrap_tool_call` | Wraps the entire tool call | Retry, approval, sandboxing |

### Key Middleware Classes
| Middleware | Purpose |
|-----------|--------|
| `ModelRetryMiddleware` | Retries model calls on transient failures |
| `ModelFallbackMiddleware` | Falls back to alternate model on failure |
| `ModelCallLimitMiddleware` | Limits max number of model calls |
| `ToolRetryMiddleware` | Retries tool calls on failure |
| `ToolCallLimitMiddleware` | Limits max number of tool calls |
| `HumanInTheLoopMiddleware` | Requires human approval for tool calls |
| `PIIMiddleware` | Detects and redacts PII in messages |
| `SummarizationMiddleware` | Summarizes long conversation history |
| `ContextEditingMiddleware` | Edits/trims context before model calls |
| `ShellToolMiddleware` | Adds shell command execution capability |
| `TodoListMiddleware` | Adds todo list management to agents |

In [0]:
import os
from langchain.agents.middleware import (
    ModelRetryMiddleware,
    ModelFallbackMiddleware,
    ModelCallLimitMiddleware,
    ToolRetryMiddleware,
    ToolCallLimitMiddleware,
    HumanInTheLoopMiddleware,
    PIIMiddleware,
    SummarizationMiddleware,
    ContextEditingMiddleware,
    RedactionRule,
)
from langchain.agents import create_agent
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool

# -----------------------------------------------------------
# Define a simple model (Claude) and a sample tool for demos
# -----------------------------------------------------------
model = ChatAnthropic(model="claude-sonnet-4-20250514")

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    # Simulated weather data
    weather_data = {
        "new york": "72°F, Sunny",
        "london": "58°F, Cloudy",
        "tokyo": "68°F, Partly Cloudy",
    }
    return weather_data.get(city.lower(), f"Weather data not available for {city}")

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

tools = [get_weather, calculate]
print("Common setup complete: model, tools defined.")

## 1. ModelRetryMiddleware

**Purpose**: Automatically retries model (LLM) calls when they fail due to transient errors (e.g., rate limits, network timeouts).

**Key Parameters**:
- `max_retries` — Maximum number of retry attempts (default: 3)
- `retry_delay` — Delay between retries in seconds
- `retry_on` — List of exception types to retry on

**When to use**: When calling external LLM APIs that may have intermittent failures or rate limits.

In [0]:
# ModelRetryMiddleware: Retries model calls on transient failures

retry_middleware = ModelRetryMiddleware(
    max_retries=3,        # Retry up to 3 times
)

# Create agent with retry middleware
agent_with_retry = create_agent(
    model,
    tools,
    middleware=[retry_middleware],
)

# Run the agent - if the model call fails, it will automatically retry
result = agent_with_retry.invoke(
    "What is the weather in New York?"
)
print(result["output"])

## 2. ModelFallbackMiddleware

**Purpose**: If the primary model fails, automatically falls back to an alternative (backup) model.

**Key Parameters**:
- `fallback_model` — The backup model to use when the primary fails
- `fallback_on` — Exception types that trigger the fallback

**When to use**: In production systems where you need high availability and can't afford downtime if one model provider goes down.

In [0]:
# ModelFallbackMiddleware: Falls back to a backup model on failure

# Define a backup model (could be a different provider or model)
backup_model = ChatAnthropic(model="claude-sonnet-4-20250514")

fallback_middleware = ModelFallbackMiddleware(
    fallback_model=backup_model,
)

agent_with_fallback = create_agent(
    model,
    tools,
    middleware=[fallback_middleware],
)

result = agent_with_fallback.invoke(
    "What is 25 * 4 + 10?"
)
print(result["output"])

## 3. ModelCallLimitMiddleware

**Purpose**: Limits the maximum number of LLM calls an agent can make in a single invocation. Prevents runaway agents that get stuck in loops.

**Key Parameters**:
- `max_calls` — Maximum number of model calls allowed

**When to use**: To prevent cost overruns and infinite loops. Essential for production deployments where you want to cap spending per request.

In [0]:
# ModelCallLimitMiddleware: Caps the number of LLM calls per invocation

call_limit_middleware = ModelCallLimitMiddleware(
    max_calls=5,  # Agent can make at most 5 LLM calls
)

agent_with_limit = create_agent(
    model,
    tools,
    middleware=[call_limit_middleware],
)

result = agent_with_limit.invoke(
    "What's the weather in New York, London, and Tokyo?"
)
print(result["output"])

## 4. ToolRetryMiddleware

**Purpose**: Automatically retries tool calls when they fail. Useful when tools depend on external services that may be temporarily unavailable.

**Key Parameters**:
- `max_retries` — Maximum retry attempts per tool call
- `retry_on` — Exception types that trigger a retry

**When to use**: When your tools call external APIs (databases, REST services) that may have intermittent failures.

In [0]:
# ToolRetryMiddleware: Retries tool calls on failure

tool_retry_middleware = ToolRetryMiddleware(
    max_retries=2,  # Retry failed tool calls up to 2 times
)

agent_with_tool_retry = create_agent(
    model,
    tools,
    middleware=[tool_retry_middleware],
)

result = agent_with_tool_retry.invoke(
    "What is the weather in London?"
)
print(result["output"])

## 5. ToolCallLimitMiddleware

**Purpose**: Limits the total number of tool calls an agent can make per invocation. Prevents agents from calling tools excessively.

**Key Parameters**:
- `max_calls` — Maximum number of tool calls allowed

**When to use**: To control costs and prevent infinite tool-calling loops. Pairs well with `ModelCallLimitMiddleware` for comprehensive guardrails.

In [0]:
# ToolCallLimitMiddleware: Caps the total number of tool calls

tool_limit_middleware = ToolCallLimitMiddleware(
    max_calls=3,  # Allow at most 3 tool calls per invocation
)

agent_with_tool_limit = create_agent(
    model,
    tools,
    middleware=[tool_limit_middleware],
)

result = agent_with_tool_limit.invoke(
    "Get weather for New York, London, and Tokyo, then calculate 100/3"
)
print(result["output"])

## 6. HumanInTheLoopMiddleware

**Purpose**: Pauses agent execution and asks for human approval before executing tool calls. Critical for high-stakes operations where you want a human to verify before actions are taken.

**Key Parameters**:
- `tools_requiring_approval` — List of tool names that need human approval (if empty, all tools require approval)
- `approve_by_default` — Whether to auto-approve if no human responds

**When to use**: For sensitive operations like database writes, financial transactions, sending emails, or any destructive action.

> **Note**: This middleware uses LangGraph's interrupt mechanism. It pauses the graph, waits for human input, and resumes.

In [0]:
# HumanInTheLoopMiddleware: Requires human approval for tool calls
# NOTE: This uses LangGraph interrupts - in a notebook, we show the setup pattern

hitl_middleware = HumanInTheLoopMiddleware()

agent_with_hitl = create_agent(
    model,
    tools,
    middleware=[hitl_middleware],
)

# In a real application, this would pause and wait for human input.
# With a checkpointer (memory), you can resume after approval:
#
# from langgraph.checkpoint.memory import MemorySaver
# checkpointer = MemorySaver()
# agent_with_hitl = create_agent(
#     model, tools,
#     middleware=[hitl_middleware],
#     checkpointer=checkpointer,
# )
#
# # First invoke pauses at tool call:
# result = agent_with_hitl.invoke(
#     "What is the weather in Tokyo?",
#     config={"configurable": {"thread_id": "1"}}
# )
# # Then resume with approval:
# from langchain.agents.middleware import InterruptOnConfig
# result = agent_with_hitl.invoke(
#     None,  # Resume with no new input
#     config={"configurable": {"thread_id": "1"}}
# )

print("HumanInTheLoopMiddleware configured (see comments for full usage pattern)")

## 7. PIIMiddleware

**Purpose**: Detects and redacts Personally Identifiable Information (PII) from messages before they are sent to the model. Helps ensure compliance with privacy regulations.

**Key Parameters**:
- `redaction_rules` — List of `RedactionRule` objects defining patterns to detect and redact
- `on_pii_detected` — Action to take: `"redact"` (replace with placeholder) or `"raise"` (throw error)

**When to use**: When handling user data that may contain SSNs, credit card numbers, emails, phone numbers, or other sensitive information.

In [0]:
# PIIMiddleware: Detects and redacts PII from messages
import re

# Define custom redaction rules using regex patterns
redaction_rules = [
    RedactionRule(
        name="email",
        pattern=re.compile(r"[\w.-]+@[\w.-]+\.\w+"),
        replacement="[REDACTED_EMAIL]",
    ),
    RedactionRule(
        name="phone",
        pattern=re.compile(r"\b\d{3}[-.]?\d{3}[-.]?\d{4}\b"),
        replacement="[REDACTED_PHONE]",
    ),
    RedactionRule(
        name="ssn",
        pattern=re.compile(r"\b\d{3}-\d{2}-\d{4}\b"),
        replacement="[REDACTED_SSN]",
    ),
]

pii_middleware = PIIMiddleware(
    redaction_rules=redaction_rules,
)

agent_with_pii = create_agent(
    model,
    tools,
    middleware=[pii_middleware],
)

# The PII in this message will be redacted before reaching the model
result = agent_with_pii.invoke(
    "My email is john@example.com and my phone is 555-123-4567. What is the weather in NYC?"
)
print(result["output"])

## 8. SummarizationMiddleware

**Purpose**: Automatically summarizes long conversation histories to keep the context window manageable. When the conversation exceeds a token threshold, older messages are summarized into a compact form.

**Key Parameters**:
- `max_tokens` — Token threshold that triggers summarization
- `summary_model` — Model used to generate summaries (can be different from the agent's main model)

**When to use**: For long-running agent conversations that might exceed the model's context window. Keeps costs down and prevents context overflow errors.

In [0]:
# SummarizationMiddleware: Summarizes long conversations to fit context window

summarization_middleware = SummarizationMiddleware(
    model=model,  # Model used for summarization
)

agent_with_summarization = create_agent(
    model,
    tools,
    middleware=[summarization_middleware],
)

# In long conversations, older messages are automatically summarized
result = agent_with_summarization.invoke(
    "Tell me the weather in all three cities: New York, London, and Tokyo."
)
print(result["output"])

## 9. ContextEditingMiddleware

**Purpose**: Allows you to programmatically edit or transform the message context before it is sent to the model. You can trim, filter, reorder, or inject messages.

**Key Parameters**:
- `edit_fn` — A function that receives the current messages and returns modified messages

**When to use**: When you need custom control over what the model sees — injecting system prompts, removing irrelevant messages, adding metadata, or implementing custom windowing strategies.

In [0]:
# ContextEditingMiddleware: Custom context transformation before model calls
from langchain_core.messages import SystemMessage

# Define a function that edits the context before each model call
def add_system_guardrails(messages):
    """Inject a system message at the beginning to guide the agent."""
    guardrail = SystemMessage(
        content="You are a helpful weather assistant. "
                "Only answer questions about weather. "
                "For non-weather questions, politely decline."
    )
    return [guardrail] + list(messages)

context_middleware = ContextEditingMiddleware(
    edit_fn=add_system_guardrails,
)

agent_with_context = create_agent(
    model,
    tools,
    middleware=[context_middleware],
)

result = agent_with_context.invoke(
    "What is the weather in Tokyo?"
)
print(result["output"])

## 10. Combining Multiple Middleware (Stacking)

**Purpose**: Middleware can be **stacked** — you can pass multiple middleware to an agent. They execute in order:
- `before_*` hooks run **first to last**
- `after_*` hooks run **last to first** (reverse order)

This lets you build a comprehensive safety and reliability layer.

In [0]:
# Stacking multiple middleware for a production-ready agent

production_middleware = [
    ModelRetryMiddleware(max_retries=3),         # 1. Retry on model failures
    ModelCallLimitMiddleware(max_calls=10),       # 2. Cap model calls
    ToolRetryMiddleware(max_retries=2),           # 3. Retry tool failures
    ToolCallLimitMiddleware(max_calls=5),         # 4. Cap tool calls
    PIIMiddleware(redaction_rules=redaction_rules), # 5. Redact PII
]

production_agent = create_agent(
    model,
    tools,
    middleware=production_middleware,
)

result = production_agent.invoke(
    "My email is test@example.com. What's the weather in London and what is 42 * 17?"
)
print(result["output"])

## Summary: Middleware Quick Reference

| # | Middleware | Hook Type | Purpose | Key Parameter |
|---|-----------|-----------|---------|---------------|
| 1 | `ModelRetryMiddleware` | `wrap_model_call` | Retry LLM calls on failure | `max_retries` |
| 2 | `ModelFallbackMiddleware` | `wrap_model_call` | Use backup model on failure | `fallback_model` |
| 3 | `ModelCallLimitMiddleware` | `before_model` | Cap LLM call count | `max_calls` |
| 4 | `ToolRetryMiddleware` | `wrap_tool_call` | Retry tool calls on failure | `max_retries` |
| 5 | `ToolCallLimitMiddleware` | `before_model` | Cap tool call count | `max_calls` |
| 6 | `HumanInTheLoopMiddleware` | `wrap_tool_call` | Human approval for tools | `tools_requiring_approval` |
| 7 | `PIIMiddleware` | `before_model` | Redact sensitive data | `redaction_rules` |
| 8 | `SummarizationMiddleware` | `before_model` | Compress long history | `max_tokens` |
| 9 | `ContextEditingMiddleware` | `before_model` | Custom context transforms | `edit_fn` |

**Best Practice**: Stack middleware in this order for production agents:
1. **Retry** (innermost - handles transient errors)
2. **Limits** (prevent runaway costs)
3. **PII/Security** (data protection)
4. **Context management** (optimize prompts)
5. **Human-in-the-loop** (outermost - final approval)

---
# Deep Dive: How to Use Middleware Hooks

Every middleware uses **hooks** — methods that fire at specific points in the agent lifecycle. There are **two ways** to create hooks:

| Approach | When to Use | Syntax |
|----------|------------|--------|
| **Decorator** (`@before_model`, etc.) | Quick, standalone, one-hook middleware | `@before_model` on a function |
| **Class-based** (subclass `AgentMiddleware`) | Complex middleware with multiple hooks, shared state | Override methods in a class |

### Hook Signatures

| Hook | Signature | Returns |
|------|-----------|--------|
| `before_agent` | `(state, runtime) -> dict \| None` | State updates before agent starts |
| `before_model` | `(state, runtime) -> dict \| None` | State updates before each LLM call |
| `after_model` | `(state, runtime) -> dict \| None` | State updates after each LLM response |
| `after_agent` | `(state, runtime) -> dict \| None` | State updates after agent completes |
| `wrap_model_call` | `(request, handler) -> ModelResponse` | Full control over the model call |
| `wrap_tool_call` | `(request, handler) -> ToolMessage` | Full control over the tool call |

### Execution Order (with 2 middleware: A, B)
```
1. A.before_agent  →  B.before_agent           (first to last)
2.   A.before_model  →  B.before_model         (first to last)
3.     A.wrap_model_call wraps B.wrap_model_call  (A=outer, B=inner)
4.   B.after_model   →  A.after_model           (last to first)
5. B.after_agent   →  A.after_agent             (last to first)
```

### Key Objects
- **`state`** — The `AgentState` dict with `messages` key (list of all messages)
- **`runtime`** — Has `stream_writer()` for emitting custom events, plus config access
- **`request`** — In wrap hooks: contains `state`, `runtime`, and call details. Use `request.override()` to modify
- **`handler`** — The callable to execute the model/tool. Call it to proceed, skip it to short-circuit, call it multiple times for retry
- **`can_jump_to`** — Special parameter: `["end"]`, `["model"]`, `["tools"]` to enable conditional flow control via `{"jump_to": "end"}`

## Hook 1: `@before_agent` — Runs Once Before Agent Starts

Fires **once** at the beginning. Use for:
- Logging/tracking agent sessions
- Initializing counters or state
- Validating input before the agent loop begins
- Early termination with `can_jump_to=["end"]`

In [0]:
%pip install --upgrade databricks-langchain
%restart_python

In [0]:
from langchain.agents.middleware import before_agent, after_agent, before_model, after_model, wrap_model_call, wrap_tool_call
from langchain.agents.middleware import AgentMiddleware, AgentState
from langchain.agents import create_agent
# from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from databricks_langchain import ChatDatabricks
import time

# model = ChatAnthropic(model="claude-sonnet-4-20250514")
model = ChatDatabricks(model = "databricks-meta-llama-3-3-70b-instruct")

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    weather_data = {"new york": "72°F, Sunny", "london": "58°F, Cloudy", "tokyo": "68°F, Partly Cloudy"}
    return weather_data.get(city.lower(), f"No data for {city}")

@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

tools = [get_weather, calculate]

# ----- DECORATOR APPROACH -----
@before_agent
def log_agent_start(state: AgentState, runtime):
    """Logs when the agent starts processing."""
    msg_count = len(state["messages"])
    print(f"🚀 [before_agent] Agent starting with {msg_count} message(s)")
    print(f"   Last user message: {state['messages'][-1].content[:80]}...")
    # Return None = no state changes
    return None

agent = create_agent(model, tools, middleware=[log_agent_start])
result = agent.invoke({"messages": [{"role": "user", "content": "What is the weather in London?"}]})
print(f"\n✅ Result: {result['messages'][-1].content}")

### Class-Based `before_agent` — With Conditional Early Exit

Using `@hook_config(can_jump_to=["end"])` allows returning `{"jump_to": "end"}` to **skip the entire agent loop**.

In [0]:
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config

# ----- CLASS-BASED APPROACH -----
class InputValidationMiddleware(AgentMiddleware):
    """Validates input before agent runs. Exits early if input is too short."""
    
    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime):
        last_msg = state["messages"][-1].content
        if len(last_msg.strip()) < 3:
            print(f"⛔ [before_agent] Input too short: '{last_msg}' — exiting early")
            from langchain_core.messages import AIMessage
            return {"messages": [AIMessage(content="Please provide a more detailed question.")], "jump_to": "end"}
        print(f"✅ [before_agent] Input validated: '{last_msg[:50]}...'")
        return None

agent = create_agent(model, tools, middleware=[InputValidationMiddleware()])

# Test with valid input
print("--- Test 1: Valid input ---")
result = agent.invoke("What is the weather in Tokyo?")
print(f"Result: {result['messages'][-1].content}\n")

# Test with too-short input (will exit early)
print("--- Test 2: Short input ---")
result = agent.invoke("Hi")
print(f"Result: {result['messages'][-1].content}")

## Hook 2: `@before_model` — Runs Before Each LLM Call

Fires **every time** the model is about to be called (can be multiple times per agent run). Use for:
- Logging each model invocation
- Injecting system prompts or context into messages
- Tracking call count / token budget
- Conditionally redirecting flow with `can_jump_to`

In [0]:
# ----- DECORATOR APPROACH: @before_model -----
@before_model
def log_each_model_call(state: AgentState, runtime):
    """Logs details before each LLM call."""
    msg_count = len(state["messages"])
    last_msg = state["messages"][-1]
    print(f"🧠 [before_model] Call #{msg_count} messages in context")
    print(f"   Last message type: {type(last_msg).__name__}")
    print(f"   Content preview: {str(last_msg.content)[:80]}")
    return None  # No state modifications

agent = create_agent(model, tools, middleware=[log_each_model_call])
result = agent.invoke("What is 25 * 4 and the weather in New York?")
print(f"\n✅ Final: {result['messages'][-1].content}")

### Class-Based `before_model` — Inject System Prompt + Call Counter

In [0]:
from langchain_core.messages import SystemMessage

# ----- CLASS-BASED: before_model with state tracking -----
class CallCounterMiddleware(AgentMiddleware):
    """Counts model calls and injects a system message."""
    
    def __init__(self):
        self._call_count = 0
    
    def before_model(self, state: AgentState, runtime):
        self._call_count += 1
        print(f"📊 [before_model] This is model call #{self._call_count}")
        
        # Inject a system prompt into the messages
        system_msg = SystemMessage(
            content=f"You are a helpful assistant. This is model call #{self._call_count}. Be concise."
        )
        # Prepend system message to the conversation
        return {"messages": [system_msg] + list(state["messages"])}

counter_mw = CallCounterMiddleware()
agent = create_agent(model, tools, middleware=[counter_mw])
result = agent.invoke("What is the weather in London and Tokyo?")
print(f"\n✅ Total model calls: {counter_mw._call_count}")
print(f"Result: {result['messages'][-1].content}")

## Hook 3: `@after_model` — Runs After Each LLM Response

Fires **after the model returns** each response. Use for:
- Logging/auditing model responses
- Detecting tool calls in the response
- Modifying or validating the AI response
- Streaming status updates to the UI

In [0]:
# ----- DECORATOR: @after_model -----
@after_model
def log_model_response(state: AgentState, runtime):
    """Inspects the model's response after each call."""
    last_msg = state["messages"][-1]
    has_tool_calls = hasattr(last_msg, "tool_calls") and last_msg.tool_calls
    
    if has_tool_calls:
        tool_names = [tc["name"] for tc in last_msg.tool_calls]
        print(f"🔧 [after_model] Model wants to call tools: {tool_names}")
    else:
        print(f"💬 [after_model] Model gave final answer: {str(last_msg.content)[:80]}...")
    return None

agent = create_agent(model, tools, middleware=[log_model_response])
result = agent.invoke("Calculate 99 * 77 and tell me the weather in Tokyo")
print(f"\n✅ Final: {result['messages'][-1].content}")

## Hook 4: `@after_agent` — Runs Once After Agent Completes

Fires **once** at the end. Use for:
- Logging completion metrics (duration, message count)
- Cleanup resources
- Sending notifications
- Final state validation

In [0]:
# ----- DECORATOR: @after_agent -----
@after_agent
def log_agent_end(state: AgentState, runtime):
    """Logs summary after agent finishes."""
    total_msgs = len(state["messages"])
    # Count tool messages (tool results)
    tool_msgs = sum(1 for m in state["messages"] if type(m).__name__ == "ToolMessage")
    print(f"🏁 [after_agent] Agent completed!")
    print(f"   Total messages: {total_msgs}")
    print(f"   Tool calls made: {tool_msgs}")
    print(f"   Final answer: {state['messages'][-1].content[:100]}...")
    return None

agent = create_agent(model, tools, middleware=[log_agent_end])
result = agent.invoke("What is 10 + 20?")
print(f"\nDone!")

## Hook 5: `@wrap_model_call` — Full Control Over Model Execution

The most powerful model hook. It **wraps** the entire model call, giving you a `handler` function to call (or not). Use for:
- **Retry logic** — Call `handler(request)` multiple times
- **Fallback** — Catch errors, switch to a different model via `request.override(model=...)`
- **Short-circuit** — Return an `AIMessage` or `ModelResponse` directly without calling handler
- **Response rewriting** — Modify the response before returning
- **Caching** — Check cache before calling, store result after

**Signature**: `wrap_model_call(request, handler) -> ModelResponse | AIMessage`
- `request` — Contains `request.state`, `request.runtime`, and model details. Use `request.override()` to modify.
- `handler` — The callable that actually executes the model. Call it to proceed.

In [0]:
from langchain.agents.middleware import ModelRequest, ModelResponse
from langchain_core.messages import AIMessage
import time

# ----- DECORATOR: @wrap_model_call with retry + timing -----
@wrap_model_call
def timed_retry_model(request: ModelRequest, handler) -> ModelResponse:
    """Retries model calls up to 3 times and measures latency."""
    max_retries = 3
    for attempt in range(max_retries):
        try:
            start = time.time()
            response = handler(request)  # <-- This calls the actual model
            elapsed = time.time() - start
            print(f"⏱️  [wrap_model_call] Model responded in {elapsed:.2f}s (attempt {attempt+1})")
            return response
        except Exception as e:
            print(f"⚠️  [wrap_model_call] Attempt {attempt+1} failed: {e}")
            if attempt == max_retries - 1:
                # Final attempt failed - return a fallback message
                return AIMessage(content="Sorry, I'm having trouble processing your request.")

agent = create_agent(model, tools, middleware=[timed_retry_model])
result = agent.invoke("What is the weather in New York?")
print(f"\n✅ Result: {result['messages'][-1].content}")

### Class-Based `wrap_model_call` — Response Rewriting + Caching

In [0]:
# ----- CLASS-BASED: wrap_model_call with response modification -----
class ResponseRewriterMiddleware(AgentMiddleware):
    """Wraps model calls to uppercase the final text response."""
    
    def __init__(self):
        self._cache = {}  # Simple in-memory cache
    
    def wrap_model_call(self, request, handler):
        # Check cache (using last message content as key)
        last_msg = request.state["messages"][-1].content
        cache_key = hash(last_msg)
        
        if cache_key in self._cache:
            print("📦 [wrap_model_call] Cache HIT - returning cached response")
            return self._cache[cache_key]
        
        print("🔄 [wrap_model_call] Cache MISS - calling model...")
        response = handler(request)  # Call the actual model
        
        # Log the response type
        ai_msg = response.result[0]
        has_tools = hasattr(ai_msg, "tool_calls") and ai_msg.tool_calls
        if not has_tools:
            print(f"📝 [wrap_model_call] Got text response, caching it")
            self._cache[cache_key] = response
        else:
            print(f"🔧 [wrap_model_call] Got tool call response, not caching")
        
        return response

agent = create_agent(model, tools, middleware=[ResponseRewriterMiddleware()])

# First call - cache miss
print("--- Call 1 (cache miss) ---")
result = agent.invoke("What is the weather in London?")
print(f"Result: {result['messages'][-1].content}")

## Hook 6: `@wrap_tool_call` — Full Control Over Tool Execution

Wraps each individual tool call. You get a `handler` to execute the tool. Use for:
- **Retry failed tools** — Call `handler(request)` multiple times
- **Modify tool arguments** — Use `request.override(tool_call=...)` to change args
- **Tool-level caching** — Cache tool results
- **Approval gate** — Check permissions before executing
- **Logging/monitoring** — Track tool usage, latency, errors

**Signature**: `wrap_tool_call(request, handler) -> ToolMessage | Command`
- `request` — Contains `request.tool_call` (dict with `name`, `args`, `id`), `request.tool` (BaseTool), `request.state`, `request.runtime`
- `handler` — Callable to execute the tool

In [0]:
from langchain_core.messages import ToolMessage

# ----- DECORATOR: @wrap_tool_call with logging + arg modification -----
@wrap_tool_call
def logged_tool_call(request, handler):
    """Logs every tool call with timing, and normalizes city names."""
    tool_name = request.tool_call["name"]
    tool_args = request.tool_call["args"]
    tool_id = request.tool_call["id"]
    
    print(f"🔧 [wrap_tool_call] Calling '{tool_name}' with args: {tool_args}")
    
    # Example: Normalize city names to lowercase for get_weather
    if tool_name == "get_weather" and "city" in tool_args:
        modified_call = {
            **request.tool_call,
            "args": {**tool_args, "city": tool_args["city"].lower().strip()}
        }
        request = request.override(tool_call=modified_call)
        print(f"   → Normalized city to: '{request.tool_call['args']['city']}'")
    
    start = time.time()
    try:
        result = handler(request)  # <-- Execute the tool
        elapsed = time.time() - start
        print(f"   ✅ '{tool_name}' completed in {elapsed:.4f}s")
        if isinstance(result, ToolMessage):
            print(f"   📤 Result: {str(result.content)[:100]}")
        return result
    except Exception as e:
        elapsed = time.time() - start
        print(f"   ❌ '{tool_name}' failed after {elapsed:.4f}s: {e}")
        # Return error as ToolMessage instead of raising
        return ToolMessage(
            content=f"Error calling {tool_name}: {e}",
            tool_call_id=tool_id
        )

agent = create_agent(model, tools, middleware=[logged_tool_call])
result = agent.invoke("Get weather for NEW YORK and calculate 2**10")
print(f"\n✅ Final: {result['messages'][-1].content}")

## Putting It All Together: All 6 Hooks in One Agent

This example shows **all hooks firing** in a single agent run, so you can see the execution order.

In [0]:
# === ALL 6 HOOKS COMBINED: See the full lifecycle ===

class FullLifecycleMiddleware(AgentMiddleware):
    """Demonstrates all 6 hooks in action with print statements."""
    
    def before_agent(self, state, runtime):
        print("\n" + "="*60)
        print("① BEFORE_AGENT: Agent is starting up")
        print(f"   Input: {state['messages'][-1].content}")
        print("="*60)
        return None
    
    def before_model(self, state, runtime):
        print(f"\n  ② BEFORE_MODEL: About to call LLM ({len(state['messages'])} msgs in context)")
        return None
    
    def wrap_model_call(self, request, handler):
        print(f"    ③ WRAP_MODEL_CALL: Executing model call...")
        start = time.time()
        response = handler(request)
        elapsed = time.time() - start
        ai_msg = response.result[0]
        has_tools = hasattr(ai_msg, "tool_calls") and ai_msg.tool_calls
        print(f"    ③ WRAP_MODEL_CALL: Done in {elapsed:.2f}s | Tool calls: {has_tools}")
        return response
    
    def after_model(self, state, runtime):
        last_msg = state["messages"][-1]
        msg_type = "tool_call" if hasattr(last_msg, "tool_calls") and last_msg.tool_calls else "text"
        print(f"  ④ AFTER_MODEL: Model responded with {msg_type}")
        return None
    
    def wrap_tool_call(self, request, handler):
        name = request.tool_call["name"]
        print(f"      ⑤ WRAP_TOOL_CALL: Executing tool '{name}'...")
        result = handler(request)
        print(f"      ⑤ WRAP_TOOL_CALL: Tool '{name}' returned: {str(result.content)[:60]}")
        return result
    
    def after_agent(self, state, runtime):
        print(f"\n{'='*60}")
        print(f"⑥ AFTER_AGENT: Agent finished!")
        print(f"   Total messages: {len(state['messages'])}")
        print(f"   Final answer: {state['messages'][-1].content[:80]}...")
        print(f"{'='*60}\n")
        return None

agent = create_agent(
    model, tools,
    middleware=[FullLifecycleMiddleware()]
)

result = agent.invoke("What is the weather in Tokyo and what is 42 * 17?")

Control the number of agent call using decorator

In [0]:
from langchain.agents.middleware import before_model, AgentState
from langchain.agents import create_agent

MAX_CALLS = 3

@before_model
def limit_model_calls(state: AgentState, runtime):
    # Count how many times the model has been called so far
    call_count = state.get("call_count", 0) + 1
    if call_count > MAX_CALLS:
        from langchain_core.messages import AIMessage
        # Stop the agent by jumping to end and returning a message
        return {
            "jump_to": "end",
            "messages": [AIMessage(content="Model call limit reached.")]
        }
    # Update the call count in state
    return {"call_count": call_count}

agent = create_agent(model, tools, middleware=[limit_model_calls])
result = agent.invoke("Ask the agent to do something complex.")
print(result["messages"][-1].content)

Control the number of agent calls using class

In [0]:
from langchain.agents.middleware import AgentMiddleware, AgentState

class CallLimitMiddleware(AgentMiddleware):
    def __init__(self, max_calls=3):
        self.max_calls = max_calls

    def before_model(self, state: AgentState, runtime):
        call_count = state.get("call_count", 0) + 1
        if call_count > self.max_calls:
            from langchain_core.messages import AIMessage
            return {
                "jump_to": "end",
                "messages": [AIMessage(content="Model call limit reached.")]
            }
        return {"call_count": call_count}

agent = create_agent(model, tools, middleware=[CallLimitMiddleware(max_calls=3)])
result = agent.invoke("Ask the agent to do something complex.")
print(result["messages"][-1].content)